In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pdfplumber
import re

def load_pdf_text(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + " "
    return re.sub(r"\s+", " ", text)

text = load_pdf_text("transformers.pdf")
print("Total characters:", len(text))
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return "".join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
class GPTDataset(Dataset):
    def __init__(self, data, block_size=64):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]
        return x, y

dataset = GPTDataset(data)
loader = DataLoader(dataset, batch_size=16, shuffle=True)
class GPT(nn.Module):
    def __init__(self, vocab_size, block_size=64, d_model=128, n_heads=4, n_layers=2):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4*d_model,
            activation="relu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

        self.block_size = block_size

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)

        x = self.token_emb(x) + self.pos_emb(pos)
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool().to(x.device)

        x = self.transformer(x, mask=mask)
        return self.lm_head(x)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GPT(vocab_size).to(device)

optimizer = optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")
def generate(model, prompt, max_new_tokens=200):
    model.eval()
    tokens = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        tokens_cond = tokens[:, -model.block_size:]
        with torch.no_grad():
            logits = model(tokens_cond)
        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        tokens = torch.cat([tokens, next_token], dim=1)

    return decode(tokens[0].tolist())
prompt = "This document explains"
output = generate(model, prompt)
print(output)



Total characters: 54573
